In [46]:
from sentence_transformers import CrossEncoder
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
import joblib

### Loading the Churn Model and Indexes

In [47]:
import pandas as pd
import numpy as np
import joblib
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# ── Load all saved components ──
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
churn_model = joblib.load('models/churn_model.pkl')
scaler = joblib.load('models/scaler.pkl')
label_encoders = joblib.load('models/label_encoders.pkl')

faq_index = faiss.read_index('indexes/faiss_faq_index')
product_index = faiss.read_index('indexes/faiss_product_index')

with open('indexes/faq_chunks.pkl', 'rb') as f:
    faq_chunks = pickle.load(f)
with open('indexes/product_chunks.pkl', 'rb') as f:
    product_chunks = pickle.load(f)

# ── Load customer lookup table ──
customers = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# ── Preprocessing config ──
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}

three_val_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
three_val_map = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}

multi_class_cols = ['InternetService', 'Contract', 'PaymentMethod']
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# ── predict_churn function ──
def predict_churn(customer_id):
    row = customers[customers['customerID'] == customer_id].copy()
    
    if row.empty:
        return {"churn_probability": 0.5, "risk_level": "MEDIUM"}
    
    row = row.drop(columns=['customerID', 'Churn'])
    row[binary_cols] = row[binary_cols].replace(binary_map)
    row[three_val_cols] = row[three_val_cols].replace(three_val_map)
    
    row['TotalCharges'] = pd.to_numeric(row['TotalCharges'], errors='coerce')
    row['tenure'] = pd.to_numeric(row['tenure'], errors='coerce')
    row['MonthlyCharges'] = pd.to_numeric(row['MonthlyCharges'], errors='coerce')
    
    for col in multi_class_cols:
        row[col] = label_encoders[col].transform(row[col])
    
    row[numeric_cols] = scaler.transform(row[numeric_cols])
    
    prob = float(churn_model.predict_proba(row)[0][1])
    risk = "HIGH" if prob > 0.6 else "MEDIUM" if prob > 0.3 else "LOW"
    
    return {"churn_probability": round(prob, 2), "risk_level": risk}

# ── Quick test ──
sample_id = customers['customerID'].iloc[0]
print(predict_churn(sample_id))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4916.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'churn_probability': 0.84, 'risk_level': 'HIGH'}


C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\4267387976.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[binary_cols] = row[binary_cols].replace(binary_map)
C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\4267387976.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[three_val_cols] = row[three_val_cols].replace(three_val_map)


In [48]:
def retrieve_chunks(query, index, chunks, top_k=3):
    query_vector = embed_model.encode([query])
    distances, indices = index.search(
        np.array(query_vector, dtype='float32'), top_k
    )
    return [chunks[i] for i in indices[0]]

### Tools

In [49]:
from langchain.tools import tool

@tool
def faq_search_tool(query: str) -> str:
    """Use for delivery, policy, refund, account, subscription, 
    Clubcard, and order queries."""
    results = retrieve_chunks(query, faq_index, faq_chunks)
    return "\n\n".join(results)

@tool
def product_search_tool(query: str) -> str:
    """Use for product suggestions, finding items, price-based 
    searches, and product availability queries."""
    results = retrieve_chunks(query, product_index, product_chunks)
    return "\n\n".join(results)

### System Prompt

In [50]:
SYSTEM_PROMPT = """You are a helpful Tesco customer support agent.

Customer ID: {customer_id}
Churn Risk: {risk_level} (Probability: {churn_prob})

Behaviour rules:
- LOW risk: Give a clear, informative response.
- MEDIUM risk: Be friendly and supportive. Mention Clubcard benefits where relevant.
- HIGH risk: Be apologetic and empathetic. Offer a concrete retention 
  incentive (delivery credit or discount). End response with: 
  RETENTION_ACTION_REQUIRED

Only use the tools provided. Do not make up information not found 
in the tools. If you cannot find the answer, say so honestly."""

In [2]:
import os
from dotenv import load_dotenv

# Load the variables from the .env file
load_dotenv()

# Access the keys
api_key = os.getenv("GOOGLE_API_KEY")
# db_pass = os.getenv("DATABASE_PASSWORD")

# Verify it worked (without printing the whole key for safety)
print(f"Key loaded: {api_key[:5]}...")

Key loaded: AIzaS...


### Agent

In [61]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
# from churn_model import predict_churn

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # fast and cheap — good for development
    temperature=0.1
)
tools = [faq_search_tool, product_search_tool]

def run_agent(customer_id: str, query: str = None):
    # Step 1: Churn prediction (Unchanged)
    churn_result = predict_churn(customer_id)
    prob = churn_result['churn_probability']
    risk = churn_result['risk_level']

    # Step 2: Proactive retention if no query (Unchanged)
    if not query:
        proactive_prompt = f"""
        Customer {customer_id} has a churn probability of {prob} (HIGH risk).
        Generate a proactive retention message offering:
        - A 10% grocery coupon
        - Highlight of Clubcard loyalty benefits
        """
        response = llm.invoke(proactive_prompt)
        return {
            "customer_id": customer_id,
            "churn_probability": prob,
            "risk_level": risk,
            "response": response.content,
            "tool_used": "None — proactive retention"
        }

    # Step 3: Build the System Message
    system_message = SYSTEM_PROMPT.format(
        customer_id=customer_id,
        risk_level=risk,
        churn_prob=prob
    )

    # Step 4: Create the LangGraph agent
    # create_react_agent handles the prompt template and tool scratchpad automatically
    agent_executor = create_react_agent(
        model=llm,
        tools=tools,
        prompt=system_message # This injects your system prompt
    )

    # Step 5: Run
    # LangGraph operates on a list of messages rather than a single "input" string
    response = agent_executor.invoke({"messages": [("user", query)]})

    # Extract tool results (retrieved chunks)
    retrieved_contexts = [
        msg.content for msg in response["messages"]
        if type(msg).__name__ == "ToolMessage"
    ]

    # The response is a dictionary containing the entire message history.
    # The final output is the content of the very last message.
    raw_content = response["messages"][-1].content
    if isinstance(raw_content, list):
        # Gemini returns list of content blocks — extract text
        final_output = " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in raw_content
        )
    else:
        final_output = raw_content

    return {
        "customer_id": customer_id,
        "churn_probability": prob,
        "risk_level": risk,
        "query": query,
        "response": final_output,
        "retrieved_contexts": retrieved_contexts
    }

In [53]:
# # Scenario 1: HIGH risk — FAQ query
# print("=" * 60)
# print("SCENARIO 1: HIGH RISK — FAQ")
# result1 = run_agent(
#     customer_id=customers['customerID'].iloc[0],
#     query="Why are delivery slots always unavailable?"
# )
# print(f"Churn: {result1['churn_probability']} | Risk: {result1['risk_level']}")
# print(result1['response'])
# 



In [54]:
# # Scenario 1: HIGH risk — FAQ query
# print("=" * 60)
# print("SCENARIO 1: HIGH RISK — FAQ")
# result1 = run_agent(
#     customer_id=customers['customerID'].iloc[0],
#     query="Why are delivery slots always unavailable?"
# )
# print(f"Churn: {result1['churn_probability']} | Risk: {result1['risk_level']}")
# print(result1['response'])

# # Scenario 2: MEDIUM risk — Product query
# print("=" * 60)
# print("SCENARIO 2: MEDIUM RISK — PRODUCT")
# result2 = run_agent(
#     customer_id=customers['customerID'].iloc[10],
#     query="Suggest a low fat yogurt under £2"
# )
# print(f"Churn: {result2['churn_probability']} | Risk: {result2['risk_level']}")
# print(result2['response'])

# # Scenario 3: LOW risk — Product query
# print("=" * 60)
# print("SCENARIO 3: LOW RISK — PRODUCT")
# result3 = run_agent(
#     customer_id=customers['customerID'].iloc[50],
#     query="Do you have gluten free bread?"
# )
# print(f"Churn: {result3['churn_probability']} | Risk: {result3['risk_level']}")
# print(result3['response'])

# # Scenario 4: Proactive retention — no query
# print("=" * 60)
# print("SCENARIO 4: PROACTIVE RETENTION")
# result4 = run_agent(
#     customer_id=customers['customerID'].iloc[0]
# )
# print(f"Churn: {result4['churn_probability']} | Risk: {result4['risk_level']}")
# print(result4['response'])

### RAGAS Faithfulness

In [62]:
eval_data = []

# Scenario 1: HIGH risk — FAQ
s1 = run_agent(
    customer_id=customers['customerID'].iloc[0],
    query="Why are delivery slots always unavailable?"
)
eval_data.append(s1)

# Scenario 2: MEDIUM risk — Product
s2 = run_agent(
    customer_id=customers['customerID'].iloc[10],
    query="Suggest a low fat yogurt under £2"
)
eval_data.append(s2)

# Scenario 3: LOW risk — Product
s3 = run_agent(
    customer_id=customers['customerID'].iloc[50],
    query="Do you have gluten free bread?"
)
eval_data.append(s3)

# Verify structure
for i, sample in enumerate(eval_data):
    print(f"\nSample {i+1}:")
    print(f"Query: {sample['query']}")
    print(f"Response: {sample['response'][:100]}")
    print(f"Contexts retrieved: {len(sample['retrieved_contexts'])}")

C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\4267387976.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[binary_cols] = row[binary_cols].replace(binary_map)
C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\4267387976.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[three_val_cols] = row[three_val_cols].replace(three_val_map)
C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\2161187103.py:43: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `fro


Sample 1:
Query: Why are delivery slots always unavailable?
Response: I'm really sorry to hear you're having trouble finding available delivery slots. I understand how fr
Contexts retrieved: 1

Sample 2:
Query: Suggest a low fat yogurt under £2
Response: That's a great choice for a healthy snack! We have a few options for low-fat yogurts under £2:

*   
Contexts retrieved: 1

Sample 3:
Query: Do you have gluten free bread?
Response: I am so sorry that you're looking for gluten-free bread and I hope I can help you find what you need
Contexts retrieved: 1


In [66]:
eval_data


[{'customer_id': '7590-VHVEG',
  'churn_probability': 0.84,
  'risk_level': 'HIGH',
  'query': 'Why are delivery slots always unavailable?',
  'response': "I'm really sorry to hear you're having trouble finding available delivery slots. I understand how frustrating it can be when you can't get a convenient time for your groceries.\n\nWhile I don't have specific information on why slots might always appear unavailable in your area, I can tell you that you can book delivery and Click+Collect slots up to 3 weeks in advance. It might be helpful to try booking further ahead if possible. We offer various options, including standard, next-day, and Flexi Saver slots, which have a 4-hour window and are often cheaper.\n\nTo help with this, I'd like to offer you £5 delivery credit for your next order.\n\nRETENTION_ACTION_REQUIRED",
  'retrieved_contexts': ['Topic: Online Grocery FAQs | Subtopic: Ordering online | Question: Slot times and options | Answer: You can choose to get your shopping deliv

In [67]:
ragas_eval_data = []

for sample in eval_data:
    ragas_eval_data.append({
        "user_input": sample["query"],
        "response": sample["response"],
        "retrieved_contexts": sample["retrieved_contexts"]
    })

In [68]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

# Wrap Gemini for RAGAS
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )
)

# Build dataset
dataset = EvaluationDataset.from_list(ragas_eval_data)

# Run evaluation
results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(llm=ragas_llm)]
)

print(results)

C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\3395595693.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness
C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\3395595693.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]Exception raised in Job[0]: RuntimeError(Timeout should be used inside a task)
C:\Users\kaush\AppData\Roaming\Python\Python314\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'Faithfulness._single_turn_ascore' was never awaited
  return counter, np.nan
Exception rai

{'faithfulness': nan}


In [70]:
import nest_asyncio
nest_asyncio.apply()

from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper

ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )
)

dataset = EvaluationDataset.from_list(ragas_eval_data)

# evaluate is synchronous — no await needed
results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(llm=ragas_llm)],
    raise_exceptions=False
)

print(results)
results_df = results.to_pandas()
print(results_df[['user_input', 'faithfulness']])

C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\1728145284.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness
C:\Users\kaush\AppData\Local\Temp\ipykernel_14532\1728145284.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]Exception raised in Job[0]: RuntimeError(Timeout should be used inside a task)
Exception raised in Job[1]: RuntimeError(Timeout should be used inside a task)
Exception raised in Job[2]: RuntimeError(Timeout should be used inside a task)
Evaluating: 100%|██████████| 3/3 [00:00<00:

{'faithfulness': nan}
                                   user_input  faithfulness
0  Why are delivery slots always unavailable?           NaN
1           Suggest a low fat yogurt under £2           NaN
2              Do you have gluten free bread?           NaN
